# PopFrame: расселение и транспортная доступность
## Практикум для слушателей ДПО по транспортному планированию

**Цель:** исследовать систему расселения Ленинградской области, построить агломерации
и зоны опорных пунктов, сравнить результаты при изменении населения и времени доступности.

Модель региона загружается из файла **`Ленинградская область.pickle`**.
Он содержит готовую модель: границу региона, населённые пункты и матрицу времени между ними. Гексагональная сетка создаётся в ноутбуке.

**Порядок работы:** запускайте ячейки сверху вниз кнопкой ▶ или сочетанием Shift+Enter.
Дождитесь окончания текущей ячейки. Меняйте значения в ячейках «Параметры».
После изменения параметров заново выполните следующие ячейки. При перезапуске
среды Colab повторите установку и загрузку данных.

**Маршрут практикума**

1. Подготовка среды и знакомство с моделью.
2. Схема расселения и агломерации.
3. Сценарии изменения населения и времени доступности.
4. Выбор опорных пунктов и построение их зон.
5. Оценка расположения территории и профилей развития.
6. Анализ землепользования.
7. Сохранение результатов и самостоятельное задание.

PopFrame помогает исследовать пространственные связи между населёнными пунктами,
сравнивать варианты развития и представлять результаты в таблицах и на картах.

## 1. Подготовка среды
### Установка библиотеки
Устанавливаем PopFrame из ветки разработки **`develop`** с проверенными версиями
GeoPandas 1.0.1 и pandas 2.2.3. Установка может занять
несколько минут. Если Colab предложит перезапустить среду после установки, выполните
перезапуск и продолжите с ячейки импорта.

Ветка обновляется со временем. Версии пакетов и идентификатор коммита автоматически сохраняются
в файл параметров вместе с результатами в разделе 8.

In [ ]:
%pip install -q --upgrade "geopandas==1.0.1" "pandas==2.2.3" "popframe[notebook] @ git+https://github.com/Mvin8/PopFrame.git@develop"

### Инструменты для загрузки
`Path` помогает указать расположение файла, а `Region` загружает модель региона.
Остальные инструменты подключим по мере выполнения соответствующих разделов.

In [ ]:
from pathlib import Path
from popframe.models.region import Region

### Загрузка учебного файла
Положите **`Ленинградская область.pickle`** в текущую рабочую директорию.
В Colab загрузите файл в `/content` через панель «Файлы»; локально запустите
ноутбук из папки с учебным файлом. Затем выполните ячейку загрузки модели.

In [ ]:
DATA_DIR = Path.cwd()
MODEL_NAME = 'Ленинградская область.pickle'
MODEL_PATH = DATA_DIR / MODEL_NAME

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Положите файл {MODEL_NAME} в {DATA_DIR} и повторите ячейку.')

region_source = Region.from_pickle(str(MODEL_PATH))
print('Модель загружена:', MODEL_PATH.name)

### Подготовка модели к анализу
`prepare_for_analysis()` проверяет геометрии, население и соответствие ID в матрице
и таблице пунктов. Метод создаёт рабочую копию в локальной проекции UTM (метры)
и определяет размерные категории пунктов по населению.

Параметр `repair_geometry=True` включает исправление самопересечений границы
с сохранением её площадных частей. Исходная модель остаётся доступной отдельно.

Матрица хранит **время в минутах**. Ноль на диагонали соответствует поездке пункта
в самого себя, бесконечность обозначает недостижимую пару.
После подготовки модель готова к построению карт и расчёту сценариев.

In [ ]:
region_model = region_source.prepare_for_analysis(repair_geometry=True)
towns = region_model.get_towns_gdf()
boundary = region_model.region
travel_times = region_model.accessibility_matrix
LOCAL_CRS = region_model.crs

print('Модель готова к анализу. Проекция:', LOCAL_CRS)

### Знакомство с данными
Посмотрим крупнейшие населённые пункты и небольшой фрагмент матрицы. Строка — пункт отправления, столбец — пункт назначения; значения — минуты.

In [ ]:
print(f'Населённых пунктов: {len(towns):,}')
print(f'Население в наборе: {towns.population.sum():,} чел.')
display(towns[['id', 'name', 'population', 'level']].nlargest(15, 'population'))
display(travel_times.iloc[:5, :5])

### Обзорная схема расселения
`Region.plot()` отображает границу региона и населённые пункты.
Серые точки показывают их расположение, синий контур — границу Ленинградской области.
По карте можно проследить участки концентрации населённых пунктов и особенности их пространственного распределения.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (11, 7), 'font.size': 11})
region_model.plot(title='Система расселения Ленинградской области')
plt.tight_layout()
plt.show()

### Параметры основной практики

- `TOWN_ID`: ID исследуемого пункта; в примере — Гатчина (`1874`).
- `BASE_TIME`: базовый порог агломераций в минутах, от 50 и выше.
- `SCENARIO_TIME`: альтернативный порог для сравнения зон доступности.
- `SCENARIO_POPULATION`: население выбранного пункта в учебном сценарии.
- `ANCHOR_IDS`: ID опорных пунктов; в примере — Гатчина и Выборг.
- `ANCHOR_TIME`: порог времени для зон опорных пунктов, минуты.

**Задание:** выполните расчёт с исходными параметрами, затем измените один параметр
и сравните результаты.

In [ ]:
TOWN_ID = 1874  # Гатчина
BASE_TIME = 80
SCENARIO_TIME = 100
SCENARIO_POPULATION = 1_000_000
ANCHOR_IDS = [1874, 1694]  # Гатчина и Выборг
ANCHOR_TIME = 50

selected_town_id = TOWN_ID
selected_town = towns.loc[TOWN_ID]
scenario_population = SCENARIO_POPULATION
print(f'Исследуемый пункт: {selected_town["name"]} (ID {TOWN_ID})')
print(f'Учебный сценарий населения: {selected_town.population:,} → {SCENARIO_POPULATION:,} чел.')

### Чтение матрицы доступности
Строка исследуемого пункта показывает время **из него** в остальные пункты.
Столбец показывает время поездки в обратном направлении.
Матрица помогает найти доступные пункты и сравнить их транспортное положение.

**Вопрос:** какие пункты доступны за базовое время? Совпадают ли ближайшие по времени
пункты с ближайшими на карте?

In [ ]:
times_from_town = travel_times.loc[selected_town_id]
reachable = towns.loc[times_from_town.le(BASE_TIME)].copy()
reachable['Время, мин'] = times_from_town.reindex(reachable.index)
display(reachable[['name', 'population', 'Время, мин']].sort_values('Время, мин').head(15))
print(f'Доступно за {BASE_TIME} мин: {len(reachable)} пунктов, включая исходный.')

## 2. Схема расселения
### `PopulationFrame.build_circle_frame()`
**Вход:** модель региона с точками, населением и размерными категориями.
**Принцип:** вокруг каждой точки строится условный круг, радиус которого зависит
от корня из численности населения и категории.
**Результат:** слой полигонов, наглядно представляющий структуру расселения.

Размеры кругов помогают сопоставить населённые пункты и выделить крупные центры.
Для определения принадлежности пунктов к агломерациям используются их исходные точки.

**Задание:** найдите крупнейшие центры и участки с малым количеством населённых пунктов.

Цвета кругов обозначают размерные категории населённых пунктов: от жёлтых
сельских поселений до красных и фиолетовых городов. Категории приведены в легенде.

In [ ]:
from popframe.method.popuation_frame import PopulationFrame
from popframe.visualization import plot_settlement_scheme

population_frame = PopulationFrame(region=region_model).build_circle_frame()
plot_settlement_scheme(region_model, population_frame)
plt.tight_layout()
plt.show()

## 3. Агломерации
### `AgglomerationBuilder.get_agglomerations()`
**Вход:** населённые пункты, матрица времени и параметр `time` в минутах.
Метод выбирает центры по размерной категории и населению, находит доступные пункты,
строит вокруг них буферы по оставшемуся времени, объединяет пересечения
и формирует границы агломераций в пределах региона.

Исходный порог `time` уменьшается на 10 минут при переходе к каждой следующей
размерной категории центров. Для перевода оставшегося времени в радиус буфера
используется коэффициент **400 м/мин**. Минимальное население центра — **15 000 человек**.

**Результат:** геометрия, центры (`core_cities`), население (`population`) и уровень
агломерации (`agglomeration_level`, 1–5), определяемый численностью населения.
Полученные слои позволяют сравнить масштаб агломераций, их центры и охват населённых пунктов.

In [ ]:
from popframe.method.agglomeration import AgglomerationBuilder

base_builder = AgglomerationBuilder(region=region_model)
base_agglomerations = base_builder.get_agglomerations(time=BASE_TIME)
base_status = base_builder.evaluate_city_agglomeration_status(towns, base_agglomerations)
display(base_agglomerations.drop(columns='geometry').head(15))
display(base_status[['name', 'population', 'agglomeration_status', 'agglomeration_level']].head(15))

### Карта агломераций и принадлежность пунктов
`evaluate_city_agglomeration_status()` определяет, пересекает ли **точка** пункта
полигон агломерации, и добавляет статус: центр, участник или вне агломерации.
Таблицу статусов можно использовать для сравнения сценариев по одним и тем же ID.

Функция `plot_settlement_scheme()` использует одинаковые цвета и легенды для всех сценариев.
Редактировать её для выполнения заданий не требуется.

Уровни агломераций показаны голубым, сиреневым, оливковым, светло-жёлтым
и тёмно-зелёным цветами. Населённые пункты сохраняют цвета своих размерных категорий.

In [ ]:
plot_settlement_scheme(region_model, population_frame, base_agglomerations,
                       title=f'Агломерации и населённые пункты: {BASE_TIME} мин')
plt.tight_layout()
plt.show()

**Вопросы к результату:** какие центры образовали общую агломерацию?
Как распределены населённые пункты внутри и вне её границ?
Какие направления связи между центрами вы бы исследовали в первую очередь?

## 4. Сценарии: меняем один фактор
### Сценарий А — изменение населения
Передаём `update_df`: таблицу новых значений населения с ID пунктов в индексе.
Библиотека пересчитывает размерную категорию и строит агломерации с обновлёнными данными.
Так можно проследить, как рост выбранного пункта влияет на систему расселения
при сохранении исходной матрицы поездок.

Для каждого сценария создаём отдельный `AgglomerationBuilder`.
Базовые результаты сохраняем для последующего сравнения.

В примере, как в исходном ноутбуке, население Гатчины (`ID = 1874`)
задаётся равным **1 000 000 человек**. Пересчитываем агломерации и круги
населённых пунктов, затем сопоставляем карты в той же палитре.

In [ ]:
import pandas as pd

population_update = pd.DataFrame({'population': [scenario_population]}, index=[selected_town_id])
scenario_towns = region_model.get_update_towns_gdf(population_update)
population_builder = AgglomerationBuilder(region=region_model)
population_agglomerations = population_builder.get_agglomerations(
    update_df=population_update, time=BASE_TIME)
population_status = population_builder.evaluate_city_agglomeration_status(scenario_towns, population_agglomerations)
population_frame_scenario = PopulationFrame(region=region_model).build_circle_frame(population_update)

display(pd.DataFrame({
    'База': towns.loc[selected_town_id, ['name', 'population', 'level']],
    'Сценарий А': scenario_towns.loc[selected_town_id, ['name', 'population', 'level']],
}))

### Сценарий Б — другой порог времени
Используем исходное население и меняем параметр `time`.
Сравнение показывает, как выбранный порог влияет на границы агломераций
и количество охваченных населённых пунктов.

In [ ]:
time_builder = AgglomerationBuilder(region=region_model)
time_agglomerations = time_builder.get_agglomerations(time=SCENARIO_TIME)
time_status = time_builder.evaluate_city_agglomeration_status(towns, time_agglomerations)

### Сравнение результатов
Сопоставим количество полигонов, число пунктов внутри агломераций
и суммарное население охваченных пунктов. Население суммируется по уникальным ID.

Все карты используют одну шкалу уровней. Таблицы показывают изменения относительно
базы и помогают связать результат с изменением выбранного параметра.

In [ ]:
def summarize_scenario(areas, status):
    inside = status['agglomeration_status'].ne('Вне агломерации')
    return {'Полигонов': len(areas), 'Пунктов внутри': int(inside.sum()),
            'Население пунктов внутри, чел.': int(status.loc[inside, 'population'].sum())}

scenario_summary = pd.DataFrame({
    'База': summarize_scenario(base_agglomerations, base_status),
    'А: население': summarize_scenario(population_agglomerations, population_status),
    'Б: время': summarize_scenario(time_agglomerations, time_status),
}).T
display(scenario_summary)
display(scenario_summary.subtract(scenario_summary.loc['База']).rename_axis('Изменение относительно базы'))

status_comparison = towns[['name', 'population']].copy()
status_comparison['База'] = base_status['agglomeration_status']
status_comparison['А: население'] = population_status['agglomeration_status']
status_comparison['Б: время'] = time_status['agglomeration_status']
changed = status_comparison['База'].ne(status_comparison['А: население']) | status_comparison['База'].ne(status_comparison['Б: время'])
display(status_comparison.loc[changed].head(20))
print('Пунктов, сменивших статус хотя бы в одном сценарии:', int(changed.sum()))

for areas, circles, title in zip(
        [base_agglomerations, population_agglomerations, time_agglomerations],
        [population_frame, population_frame_scenario, population_frame],
        [f'База: {BASE_TIME} мин', f'А: население, {BASE_TIME} мин', f'Б: время, {SCENARIO_TIME} мин']):
    plot_settlement_scheme(region_model, circles, areas, title=title)
    plt.tight_layout()
    plt.show()

**Задание:** опишите изменения границ, уровней и состава агломераций.
Свяжите результат с категорией выбранного пункта, его доступностью и положением
относительно других центров. Отметьте территории, сохранившие свои характеристики.

## 5. Опорные населённые пункты
### `AnchorSettlementBuilder.get_anchor_settlement_boundaries()`
**Вход:** модель региона, слой точек с признаком `is_anchor_settlement`
и порог `time`. Признак позволяет задать исследуемый набор опорных пунктов.

Метод использует матрицу времени, строит и объединяет зоны доступности,
затем разделяет общие области, в том числе с помощью диаграммы Вороного.
Результат помогает сравнить пространственный охват разных наборов опорных пунктов.

В учебном сценарии опорными пунктами заданы Гатчина и Выборг.
Для исследования своего варианта задайте `ANCHOR_IDS` в ячейке параметров
и повторите расчёт.

Опорные пункты выделены зелёным цветом, их зоны — светло-голубым
с чёрной границей. Серые точки показывают остальные населённые пункты.

In [ ]:
from popframe.method.anchor_settlement import AnchorSettlementBuilder

anchor_towns = towns.copy()
anchor_ids = ANCHOR_IDS
anchor_towns['is_anchor_settlement'] = anchor_towns.index.isin(anchor_ids)
display(anchor_towns.loc[anchor_ids, ['id', 'name', 'population']])

anchor_boundaries = AnchorSettlementBuilder(region=region_model).get_anchor_settlement_boundaries(
    towns=anchor_towns, time=ANCHOR_TIME)

from popframe.visualization import plot_anchor_settlements

plot_anchor_settlements(region_model, anchor_boundaries, anchor_towns,
                        title=f'Зоны опорных пунктов: {ANCHOR_TIME} мин')
plt.tight_layout()
plt.show()

**Вопросы:** как соотносятся зоны опорных пунктов и агломерации?
Как изменяется охват территории при выборе другого опорного пункта?

## 6. Расположение выбранной территории
### Создание гексагональной сетки
`create_leningrad_hexagons()` строит 2148 гексагонов учебного набора по адресам H3.
Сетка имеет тот же размер ячеек, расположение и числовые ID, что и исходный слой.
Готовые полигоны создаются при запуске ячейки в системе координат региона.

Сетка позволяет выбирать сопоставимые по масштабу территории для оценки их положения
в системе расселения. По умолчанию будем исследовать **гексагон 1632**.

In [ ]:
from popframe.preprocessing.hex_grid import create_leningrad_hexagons

hexagons = create_leningrad_hexagons(crs=LOCAL_CRS)
print('Создано гексагонов:', len(hexagons))

### Интерактивная карта гексагонов
`GeoDataFrame.explore()` открывает интерактивную карту. Приблизьте нужный участок
и наведите курсор на гекс, чтобы узнать его числовой ID; при нажатии ID появится во всплывающем окне.
Впишите выбранный ID в следующую ячейку. По умолчанию используется **1632**.

In [ ]:
hexagons.explore(tiles='OpenStreetMap', tooltip=['id'], popup=['id'])

### Выбор территории по ID
`HEX_ID` задаёт числовой ID исследуемого гекса. По нему выбираем строку сетки
и передаём полигон в методы оценки территории. После изменения ID выполните
эту ячейку и все последующие ячейки.

In [ ]:
HEX_ID = 1632  # ID гекса с интерактивной карты
territory = hexagons.loc[hexagons['id'].eq(HEX_ID)].copy().reset_index(drop=True)
territory['name'] = f'Гексагон {HEX_ID}'
display(territory[['id', 'name']])
territory.explore(tiles='OpenStreetMap', tooltip=['id', 'name'])

### Выбранная территория в системе расселения
`Region.plot(territories=territory)` показывает населённые пункты, границу региона
и выбранный гекс. Обзорная карта помогает оценить положение территории относительно
центров и участков концентрации населённых пунктов.

In [ ]:
region_model.plot(territories=territory, title=f'Схема расселения: гексагон {HEX_ID}')
plt.tight_layout()
plt.show()

### `TerritoryEvaluation.evaluate_territory_location()`
Метод оценивает положение территории в системе расселения.
Сначала он находит населённые пункты в ближнем буфере и выбирает наибольший балл
их размерной категории, затем проверяет расположение территории между центрами.

**Результат:** балл, текстовое пояснение и сведения о связанных населённых пунктах.
Оценка помогает сопоставлять площадки по их пространственному положению.

**Задание:** сопоставьте пояснение с картой. Какие населённые пункты определяют
положение выбранной территории?

In [ ]:
from popframe.method.territory_evaluation import TerritoryEvaluation

evaluation = TerritoryEvaluation(region=region_model)
location_results = pd.DataFrame(evaluation.evaluate_territory_location(territory))
territory_scored = territory.copy()
territory_scored['score'] = location_results['score'].to_numpy()
display(location_results)

fig, ax = plt.subplots(figsize=(9, 7))
territory.plot(ax=ax, color='#893434', edgecolor='#893434', alpha=0.8)
nearby_towns = towns.loc[towns.intersects(territory.geometry.iloc[0].buffer(15000))]
nearby_towns.plot(ax=ax, color='#333333', markersize=20)
for row in nearby_towns.nlargest(5, 'population').itertuples():
    ax.annotate(row.name, (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points')
ax.set_title('Учебная территория и населённые пункты в её окружении')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Карта оценок всех гексагонов
Применим `evaluate_territory_location()` ко всей сетке, чтобы сравнить территории.
На интерактивной карте используется палитра **RdYlGn**: от красного для низких
положительных баллов через жёлтый к зелёному для высоких. Нулевой балл показан
чёрным, как в исходном ноутбуке. При наведении видны ID гекса и его балл.

In [ ]:
from popframe.visualization import explore_territory_scores

hexagons_scored = hexagons.copy()
grid_results = pd.DataFrame(evaluation.evaluate_territory_location(hexagons_scored))
hexagons_scored['score'] = grid_results['score'].to_numpy()
explore_territory_scores(hexagons_scored)

### `TerritoryEvaluation.calculate_potential()` — профили развития
Метод сравнивает баллы по пяти критериям с требованиями профилей развития:
жилая застройка, промышленность, рекреация и другие.

Для каждого профиля рассчитываются число выполненных критериев и взвешенное
превышение требований. По этим показателям формируется ранжированный список профилей.

В примере баллы «Население», «Транспорт», «Экология», «Соц-об» и «Инж инф»
задаются в словаре `criteria` как параметры учебного сценария.

**Задание:** увеличьте значение «Транспорт» на один балл и сравните порядок профилей.
Какие профили изменили положение в списке и какие критерии повлияли на результат?

In [ ]:
criteria = {'Население': 4, 'Транспорт': 4, 'Экология': 3, 'Соц-об': 3, 'Инж инф': 2}
ranked_profiles = evaluation.calculate_potential(criteria)
profile_table = pd.DataFrame([
    {'Профиль': name, 'Выполнено критериев': values[0], 'Взвешенное превышение': values[1]}
    for name, values in ranked_profiles
])
display(profile_table)

## 7. Землепользование
### `LandUseAssessment`
`get_landuse_data()` получает объекты OpenStreetMap в выбранном полигоне,
группирует их по типам землепользования и рассчитывает доли площади.
`plot_landuse()` отображает категории на картографической подложке.

Результат позволяет изучить состав землепользования и сопоставить его
с рассматриваемыми профилями развития территории.

Запуск следующей ячейки получает данные OSM для выбранной территории
и строит карту. Для этого используется подключение к интернету.

In [ ]:
from popframe.method.landuse_assessment import LandUseAssessment

landuse_method = LandUseAssessment(region=region_model)
landuse_data = landuse_method.get_landuse_data(territory.to_crs(4326))
display(landuse_data.drop(columns='geometry'))
landuse_method.plot_landuse(territory.to_crs(4326), landuse_data)

## 8. Сохранение результатов
Сохраняем карты-слои в GeoJSON (долгота/широта), таблицы — в CSV, а параметры,
версии и контрольную сумму учебного файла — в JSON. Это позволяет связать результат
с конкретным набором данных и условиями расчёта. Исходный pickle не изменяется.

В Colab результаты находятся в папке `/content/results`. Последняя ячейка
создаёт ZIP-архив; скачайте его через панель «Файлы» до завершения сессии.

### Сведения для воспроизводимости
Служебная ячейка собирает версии и источник установки. Они попадут в `parameters.json`; редактировать её не требуется.

In [ ]:
from importlib import metadata
import json

versions = {name: metadata.version(name) for name in
            ['popframe', 'geopandas', 'pandas', 'numpy', 'shapely', 'pydantic', 'h3', 'folium']}
direct_url = metadata.distribution('popframe').read_text('direct_url.json')
source_info = json.loads(direct_url)

### Запись файлов и создание архива
Выполните ячейку и скачайте полученный ZIP-архив.

In [ ]:
import hashlib
import shutil

RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
for filename, layer in {
    'agglomerations_base': base_agglomerations,
    'agglomerations_population': population_agglomerations,
    'agglomerations_time': time_agglomerations,
    'anchor_boundaries': anchor_boundaries,
    'territory_location': territory_scored,
    'hexagons': hexagons_scored,
}.items():
    output_path = RESULTS_DIR / f'{filename}.geojson'
    layer.to_crs(4326).to_file(output_path, driver='GeoJSON')

scenario_summary.to_csv(RESULTS_DIR / 'scenario_summary.csv', encoding='utf-8-sig')
status_comparison.to_csv(RESULTS_DIR / 'town_status.csv', encoding='utf-8-sig', index_label='town_id')
location_results.to_csv(RESULTS_DIR / 'territory_location.csv', encoding='utf-8-sig', index=False)
profile_table.to_csv(RESULTS_DIR / 'development_profiles.csv', encoding='utf-8-sig', index=False)
with MODEL_PATH.open('rb') as model_file:
    model_sha256 = hashlib.file_digest(model_file, 'sha256').hexdigest()
parameters = {'model_file': MODEL_NAME, 'model_sha256': model_sha256, 'versions': versions,
    'package_source': source_info, 'local_crs': str(LOCAL_CRS), 'town_id': int(selected_town_id),
    'base_time_min': BASE_TIME, 'scenario_time_min': SCENARIO_TIME,
    'scenario_population': scenario_population, 'anchor_ids': [int(value) for value in anchor_ids],
    'anchor_time_min': ANCHOR_TIME, 'criteria_assumptions': criteria,
    'site_name': territory.iloc[0]['name'], 'hex_id': HEX_ID, 'h3_id': territory.iloc[0]['h3_id']}
(RESULTS_DIR / 'parameters.json').write_text(json.dumps(parameters, ensure_ascii=False, indent=2), encoding='utf-8')
archive_path = shutil.make_archive(str(DATA_DIR / 'PopFrame_results'), 'zip', RESULTS_DIR)
print('Архив результатов:', archive_path)

## 9. Самостоятельная работа

1. Выберите другой населённый пункт и обоснуйте выбор.
2. Задайте сценарий изменения его населения. Сравните категорию, границы и число
   охваченных пунктов с базой; приведите значения из таблицы результатов.
3. Измените порог времени и опишите, как изменился пространственный охват агломераций.
4. Выберите другой набор опорных пунктов и сравните полученные зоны.
5. Сформулируйте планировочный вывод на основе карт и таблиц.

### Другие возможности библиотеки
`CityPopulationScorer.run()` оценивает гексагоны по населению муниципалитетов.
Для расчёта используются муниципальные полигоны с полями `territory_id`, `population`
и гексагоны с полем `hexagon_id`.

Для подготовки модели с нуля `AdjacencyCalculator.get_dataframe()` рассчитывает
матрицу доступности по транспортному графу с весами `time_min` и полигонам
с согласованными ID. Для этого расчёта подключается `networkit`.
`PopulationFiller.fill()` распределяет население исходных территорий между пунктами
по модельным весам. В данном практикуме подготовленная модель уже загружена из pickle.

### Подсказки по работе в Colab

- Учебные файлы размещаются в текущей директории — `/content` в Colab.
- После перезапуска среды повторите установку, импорт и загрузку модели.
- Для обращения к файлу используйте его полное имя, включая расширение.
- Для получения землепользования запустите ячейку раздела 7.
- Готовый архив результатов скачивается через панель «Файлы».